# 10. Ensemble of SVM and Llama on dataset C: repeated splits and a NIO control

This notebook follows the ensemble notebook (`EnsembleSVMandLLAMA32_EIGHT.ipynb`): a small neural
network takes the SVM's prediction and Llama's prediction (23 one-hot inputs) and learns the
annotated tool. It keeps that recipe and adds checks that a reviewer is likely to ask for:

1. **Reproduce the original split.** The original notebook picks its train/test split with an
   unseeded `random.random()`, so each run differs. Seed 87 gives the split it showed.
2. **Repeat over 100 random splits** and report mean, spread, and how often the ensemble wins.
   Adds a logistic regression on the same 23 inputs, and longer training (the original stops at
   200 epochs, when the loss is still falling).
3. **Check the SVM column for leakage.** `pred_svm` is out of fold over all 1500 rows, so for a test
   row the SVM that predicted it was trained on other test rows' labels. Here the SVM input is
   rebuilt using only each split's training rows, and the results are compared.
4. **NIO case study with a control.** The original NIO run on the 14 prompts where Llama answered
   `Unknown` is repeated, and compared with the same procedure on prompts where Llama did not
   answer `Unknown`. A constrained version keeps each input block one-hot-like (softmax).

Run from the repository folder (needs `cyber_dataset_C.csv`).

In [1]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

df = pd.read_csv("cyber_dataset_C.csv")
enc = LabelEncoder(); y = enc.fit_transform(df["tool_ground_truth"])
TOOLS = list(enc.classes_); K = len(TOOLS)                        # 11 tools
cid = {t: i for i, t in enumerate(TOOLS)}
svm_idx   = np.array([cid[t] for t in df["pred_svm"]])
llama_idx = np.array([cid.get(t, K) for t in df["tool_predicted"]])  # 11 = Unknown
idx = np.arange(len(df))

def features(svm_i, llama_i):
    X = np.zeros((len(svm_i), 2 * K + 1), np.float32)               # 11 SVM + 12 Llama
    X[np.arange(len(svm_i)), svm_i] = 1
    X[np.arange(len(svm_i)), K + llama_i] = 1
    return X

def train_net(X, t, epochs, seed=42):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(X.shape[1], 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, K))
    opt, loss_fn = torch.optim.Adam(net.parameters(), lr=1e-3), nn.CrossEntropyLoss()
    Xt, tt = torch.tensor(X), torch.tensor(t, dtype=torch.long)
    for _ in range(epochs):                                         # full batch, as in the original
        opt.zero_grad(); loss = loss_fn(net(Xt), tt); loss.backward(); opt.step()
    return net, loss.item()

def predict(net, X):
    with torch.no_grad():
        return torch.argmax(net(torch.tensor(X)), 1).numpy()

X_all = features(svm_idx, llama_idx)
print(len(df), "rows |", X_all.shape[1], "input features |", int((llama_idx == K).sum()), "Llama 'Unknown' answers")

1500 rows | 23 input features | 14 Llama 'Unknown' answers


## 1. Reproduce the original split

Seed 87 reproduces the test rows shown in the original notebook (its first test indices were
1348, 1221, 607, 720, 900). The network is trained for 200 epochs as in the original. The original
reported ensemble 95.3%, SVM 94.3%, Llama 87.3% on this split.

In [2]:
tr, te = train_test_split(idx, test_size=0.20, random_state=87, stratify=y)
print("first test indices:", list(te[:5]), "(original notebook: [1348, 1221, 607, 720, 900])")
net, loss = train_net(X_all[tr], y[tr], epochs=200)
print(f"original recipe on seed 87:  ensemble {(predict(net, X_all[te]) == y[te]).mean():.1%} | "
      f"SVM {(svm_idx[te] == y[te]).mean():.1%} | Llama {(llama_idx[te] == y[te]).mean():.1%} | final training loss {loss:.2f}")
print(f"SVM over all 1500 rows: {(svm_idx == y).mean():.1%}  (this split is easier for the SVM than average)")

first test indices: [1348, 1221, 607, 720, 900] (original notebook: [1348, 1221, 607, 720, 900])


original recipe on seed 87:  ensemble 95.3% | SVM 94.3% | Llama 87.3% | final training loss 0.47
SVM over all 1500 rows: 91.9%  (this split is easier for the SVM than average)


## 2. 100 random 80/20 splits

Each split is stratified on the tool. Columns: the ensemble trained for 200 epochs (original) and
1500 epochs, a logistic regression on the same 23 inputs, the SVM alone, and Llama alone
(`Unknown` counts as wrong).

In [3]:
rows = []
for s in range(100):
    tr, te = train_test_split(idx, test_size=0.20, random_state=s, stratify=y)
    n200, l200 = train_net(X_all[tr], y[tr], 200)
    n1500, l1500 = train_net(X_all[tr], y[tr], 1500)
    lr = LogisticRegression(max_iter=2000).fit(X_all[tr], y[tr])
    acc = lambda p: (p == y[te]).mean()
    rows.append({"split": s, "ensemble, 200 epochs": acc(predict(n200, X_all[te])),
                 "ensemble, 1500 epochs": acc(predict(n1500, X_all[te])),
                 "logistic regression": acc(lr.predict(X_all[te])),
                 "SVM alone": acc(svm_idx[te]), "Llama alone": acc(llama_idx[te]),
                 "loss 200": l200, "loss 1500": l1500})
R = pd.DataFrame(rows)
cols = ["ensemble, 200 epochs", "ensemble, 1500 epochs", "logistic regression", "SVM alone", "Llama alone"]
summary = pd.DataFrame({"mean %": R[cols].mean() * 100, "SD": R[cols].std() * 100,
                        "min %": R[cols].min() * 100, "max %": R[cols].max() * 100}).round(1)
print(summary.to_string()); print()
print(f"final training loss: 200 epochs {R['loss 200'].mean():.2f}, 1500 epochs {R['loss 1500'].mean():.2f}")
for name in ["ensemble, 200 epochs", "ensemble, 1500 epochs", "logistic regression"]:
    d = (R[name] - R["SVM alone"]) * 100
    print(f"{name:24} minus SVM: mean {d.mean():+.2f} points (SD {d.std():.2f}); ahead in {(d > 0).sum()}, tied in {(d == 0).sum()}, behind in {(d < 0).sum()} of 100 splits")
R.to_csv("ensemble_splits_C.csv", index=False)

                       mean %   SD  min %  max %
ensemble, 200 epochs     93.5  1.2   90.7   95.7
ensemble, 1500 epochs    94.2  1.1   91.3   97.3
logistic regression      94.2  1.1   91.3   96.7
SVM alone                91.9  1.4   88.3   94.3
Llama alone              88.8  1.6   83.7   92.3

final training loss: 200 epochs 0.50, 1500 epochs 0.09
ensemble, 200 epochs     minus SVM: mean +1.61 points (SD 1.12); ahead in 88, tied in 5, behind in 7 of 100 splits
ensemble, 1500 epochs    minus SVM: mean +2.36 points (SD 1.32); ahead in 96, tied in 3, behind in 1 of 100 splits
logistic regression      minus SVM: mean +2.36 points (SD 1.23); ahead in 98, tied in 0, behind in 2 of 100 splits


## 3. Does the out-of-fold SVM column leak?

For every split, the SVM input is rebuilt from the training rows only: training rows get 5-fold
out-of-fold SVM predictions *within the training rows*, test rows get predictions from an SVM
trained on all training rows. The test labels are never used. The ensemble is then trained and
tested as before (1500 epochs) and compared with the stored `pred_svm` version on the same splits.

In [4]:
def svm_pipeline():
    return TfidfVectorizer(ngram_range=(1, 2), max_features=5000), SVC(kernel="linear", random_state=42)

def clean_svm_predictions(tr, te):
    texts = df["prompt"].astype(str)
    train_pred = np.empty(len(tr), dtype=int)
    for a, b in StratifiedKFold(5, shuffle=True, random_state=42).split(tr, y[tr]):
        v, m = svm_pipeline(); m.fit(v.fit_transform(texts.iloc[tr[a]]), y[tr[a]])
        train_pred[b] = m.predict(v.transform(texts.iloc[tr[b]]))
    v, m = svm_pipeline(); m.fit(v.fit_transform(texts.iloc[tr]), y[tr])
    return train_pred, m.predict(v.transform(texts.iloc[te]))

out = []
for s in range(30):
    tr, te = train_test_split(idx, test_size=0.20, random_state=s, stratify=y)
    tr_svm, te_svm = clean_svm_predictions(tr, te)
    Xtr = features(tr_svm, llama_idx[tr]); Xte = features(te_svm, llama_idx[te])
    net, _ = train_net(Xtr, y[tr], 1500)
    out.append({"ensemble, clean SVM input": (predict(net, Xte) == y[te]).mean(),
                "ensemble, stored pred_svm": R.loc[s, "ensemble, 1500 epochs"],
                "SVM alone, clean": (te_svm == y[te]).mean(), "SVM alone, stored pred_svm": R.loc[s, "SVM alone"]})
L = pd.DataFrame(out)
print((L.mean() * 100).round(1).to_string())
d = (L["ensemble, stored pred_svm"] - L["ensemble, clean SVM input"]) * 100
print(f"\nstored minus clean (ensemble), 30 splits: mean {d.mean():+.2f} points, SD {d.std():.2f}")

ensemble, clean SVM input     94.3
ensemble, stored pred_svm     94.3
SVM alone, clean              91.6
SVM alone, stored pred_svm    91.4

stored minus clean (ensemble), 30 splits: mean +0.00 points, SD 1.04


## 4. NIO case study with a control

The original run: freeze the ensemble, start from random 23-dimensional inputs, and optimize them
(Adam, 2000 steps, values clamped to [0, 1]) so that the network outputs the *annotated* tool of
the 14 prompts where Llama answered `Unknown`. The question is whether the optimized inputs turn
on the `Unknown` input.

Two additions: (a) the same procedure on 14 prompts where Llama did **not** answer `Unknown`,
repeated for 20 random draws, as a control; (b) a constrained version where the SVM block and the
Llama block are each a softmax, so every input stays one-hot-like. In (b) the "result" is which
class each block picks. The network is the one trained on the seed-87 split, 1500 epochs.

In [5]:
tr87, te87 = train_test_split(idx, test_size=0.20, random_state=87, stratify=y)
net, loss = train_net(X_all[tr87], y[tr87], 1500)
for p in net.parameters(): p.requires_grad_(False)
print(f"network accuracy on its test split: {(predict(net, X_all[te87]) == y[te87]).mean():.1%} (training loss {loss:.2f})")

unknown_rows = np.flatnonzero(llama_idx == K)
print("Unknown rows:", len(unknown_rows), "| real SVM correct on", int((svm_idx[unknown_rows] == y[unknown_rows]).sum()), "of them")

def nio_free(rows, seed=42):
    torch.manual_seed(seed); X = torch.rand(len(rows), 23, requires_grad=True)
    opt, tgt = torch.optim.Adam([X], lr=0.01), torch.tensor(y[rows], dtype=torch.long)
    for _ in range(2000):
        opt.zero_grad(); nn.CrossEntropyLoss()(net(X), tgt).backward(); opt.step()
        with torch.no_grad(): X.clamp_(0, 1)
    return X.detach().numpy()

def nio_softmax(rows, seed=42):
    torch.manual_seed(seed)
    a = torch.randn(len(rows), K, requires_grad=True); b = torch.randn(len(rows), K + 1, requires_grad=True)
    opt, tgt = torch.optim.Adam([a, b], lr=0.05), torch.tensor(y[rows], dtype=torch.long)
    for _ in range(2000):
        opt.zero_grad()
        nn.CrossEntropyLoss()(net(torch.cat([a.softmax(1), b.softmax(1)], 1)), tgt).backward(); opt.step()
    return a.softmax(1).detach().numpy(), b.softmax(1).detach().numpy()

known_rows = np.flatnonzero(llama_idx < K)
controls = [np.random.RandomState(i).choice(known_rows, len(unknown_rows), replace=False) for i in range(20)]

network accuracy on its test split: 94.7% (training loss 0.09)
Unknown rows: 14 | real SVM correct on 14 of them


In [6]:
# (a) free NIO, as in the original: how often is the 'Unknown' input switched on (> 0.5)?
Xu = nio_free(unknown_rows)
ctrl_counts = [int((nio_free(r)[:, 2 * K] > 0.5).sum()) for r in controls]
print(f"free NIO, Unknown rows:   Unknown input > 0.5 in {(Xu[:, 2 * K] > 0.5).sum()}/{len(unknown_rows)}")
print(f"free NIO, 20 control draws (Llama not Unknown): Unknown input > 0.5 in {np.mean(ctrl_counts):.1f}/{len(unknown_rows)} on average (min {min(ctrl_counts)}, max {max(ctrl_counts)})")
print(f"optimized rows with 2 or more SVM inputs switched on (a real input has exactly one): {((Xu[:, :K] > 0.5).sum(1) >= 2).mean():.0%}")
print(f"share of all optimized values above 0.9 (saturated): {(Xu > 0.9).mean():.0%}")

free NIO, Unknown rows:   Unknown input > 0.5 in 9/14
free NIO, 20 control draws (Llama not Unknown): Unknown input > 0.5 in 6.8/14 on average (min 4, max 9)
optimized rows with 2 or more SVM inputs switched on (a real input has exactly one): 100%
share of all optimized values above 0.9 (saturated): 14%


In [7]:
# (b) constrained NIO: each block is a softmax, so it stays one-hot-like
a, b = nio_softmax(unknown_rows)
print(f"constrained NIO, Unknown rows: Llama block picks 'Unknown' in {(b.argmax(1) == K).sum()}/{len(unknown_rows)}; "
      f"SVM block picks the annotated tool in {(a.argmax(1) == y[unknown_rows]).sum()}/{len(unknown_rows)}")
cc_unknown, cc_svm = [], []
for r in controls:
    a_c, b_c = nio_softmax(r)
    cc_unknown.append(int((b_c.argmax(1) == K).sum())); cc_svm.append(int((a_c.argmax(1) == y[r]).sum()))
print(f"constrained NIO, 20 control draws: Llama block picks 'Unknown' in {np.mean(cc_unknown):.1f}/{len(unknown_rows)} on average "
      f"(min {min(cc_unknown)}, max {max(cc_unknown)}); SVM block picks the annotated tool in {np.mean(cc_svm):.1f}/{len(unknown_rows)}")

constrained NIO, Unknown rows: Llama block picks 'Unknown' in 0/14; SVM block picks the annotated tool in 14/14


constrained NIO, 20 control draws: Llama block picks 'Unknown' in 0.0/14 on average (min 0, max 0); SVM block picks the annotated tool in 14.0/14


## How to read this

The numbers above are printed by the code. Points to keep in mind when writing them up:

- Report the **mean and spread over many splits** (section 2), not a single split. The split in the
  original notebook is one draw and happens to be favourable to the SVM.
- The ensemble can be compared with the logistic regression: if they are close, the network is
  needed for NIO (it needs a differentiable model) and not for accuracy.
- Section 3 shows whether using out-of-fold SVM predictions for all rows changes the ensemble result.
- Section 4 is a case study on 14 prompts. Compare the Unknown rows with the control draws before
  claiming that NIO has found the `Unknown` input. In the real data the SVM was right on every one
  of these 14 prompts, so a high SVM input for the annotated tool is expected.